In [ ]:
# Settings
filepath = r"C:\Users\Jaeda\OneDrive\Downloads\PTON SENIOR YEAR ARCHIVE\xwords\pros and cons.cfp"
title = "Pros and Cons"         # FORMAT: "Pros and Cons"
weekday = "Tuesday"             # FORMAT: "Tuesday"
date = "March 23, 2026"         # FORMAT: "March 23, 2026"
author = "Jaeda Woodruff"       # FORMAT: "Jaeda Woodruff"
editor = "Jaeda Woodruff"       # FORMAT: "Jaeda Woodruff"

In [ ]:
# Import libraries
import numpy as np

# Get intermediates
full_date = weekday + " Crossword, " + date
full_byline = "By " + author + " / Edited by " + editor

# Get html filename (e.g. march-23-2026.html)
split_date = date.split(' ')
split_date_month = split_date[0].lower()
split_date_day = split_date[1][:-1]
split_date_year = split_date[2]
filename = split_date_month + '-' + split_date_day + '-' + split_date_year + '.html'

# Read file
with open(filepath, "r", encoding="utf-8") as f:
    data = f.read()

# Get n
n = data.splitlines()[6]
start = n.find('<GRID width="')
end = n.find('">')
n = int(n[start + 13:end])

# Get black cell IDs
fill = data.splitlines()[7:7+n]
fill = ''.join(fill)
blackCells = []
while ('.' in fill):
    idx = fill.find('.') + 1
    fill = fill.replace('.','-',1)
    blackCells.append(idx)

# Get across word nums
acrossWordNums = []
i = 1
temp = data.splitlines()[7+n+2]
while 'dir="ACROSS"' in temp:
    start = temp.find('num="')
    end = temp.find('">')
    temp = int(temp[start + 5:end])
    acrossWordNums.append(temp)
    temp = data.splitlines()[7+n+2+i]
    i += 1

# Get down word nums
downWordNums = []
while 'dir="DOWN"' in temp:
    start = temp.find('num="')
    end = temp.find('">')
    temp = int(temp[start + 5:end])
    downWordNums.append(temp)
    temp = data.splitlines()[7+n+2+i]
    i += 1

# Get across word IDs
temp1 = [c + 1 for c in blackCells] # after black cells
temp2 = [(n*i) + 1 for i in range(n)]  # first in row
acrossWordIDs = np.sort(list(set(temp1 + temp2))) #combine lists
# remove double features
toRemove = []
[toRemove.append(i) for i in range(len(acrossWordIDs)) if acrossWordIDs[i] in blackCells]
acrossWordIDs = [int(element) for i, element in enumerate(acrossWordIDs) if i not in toRemove]
# remove 1 if there's a blackcell in cell 1
if 1 in blackCells:
    acrossWordIDs.remove(1)

# Get down word IDs
temp1 = [c + n for c in blackCells] # below black cells
temp2 = [(i + 1) for i in range(n)] # first in column
downWordIDs = np.sort(list(set(temp1 + temp2))) #combine lists
# remove double features and anything after bottom row
toRemove = []
[toRemove.append(i) for i in range(len(downWordIDs)) if downWordIDs[i] in blackCells]
[toRemove.append(i) for i in range(len(downWordIDs)) if downWordIDs[i] > (n*n)]
downWordIDs = [int(element) for i, element in enumerate(downWordIDs) if i not in toRemove]

# Get across word clues
acrossClues = []
i = 1
temp = data.splitlines()[7+n+2]
while 'dir="ACROSS"' in temp:
    start = temp.find('">')
    end = temp.find('</WORD>')
    temp = temp[start + 2:end]
    acrossClues.append(temp)
    temp = data.splitlines()[7+n+2+i]
    i += 1

# Get down word clues
downClues = []
while 'dir="DOWN"' in temp:
    start = temp.find('">')
    end = temp.find('</WORD>')
    temp = temp[start + 2:end]
    downClues.append(temp)
    temp = data.splitlines()[7+n+2+i]
    i += 1

# Get answers
answers = "".join(data.splitlines()[7:7+n])

In [ ]:
# Create word start list
wordStartList = np.sort(list(set(acrossWordIDs + downWordIDs)))

# Create new cell elements
i = 1
a = 0
b = 0
b_alt = 0
bstring = []
divline = '<div'
divline_fill = '<div'
grid_subdivs = ''
for x in range(1, (n*n)+1):

    # Update pointers
    if x in acrossWordIDs[1:]:
        a = a + 1
    if x in downWordIDs[1:]:
        b = b + 1
    elif x != 1 and x != downWordIDs[1:] and x not in blackCells:
        b_alt = bstring[x-n-1]

    # Select cell type (cell or blackCell)
    if x in blackCells:
        cellType = 'blackCell'
    else:
        cellType = 'cell'

    # Select across number (1across etc) and down number (1down etc)
    if x in blackCells:
        acrossNumber = ''
        downNumber = ''
    else:
        acrossNumber = ' ' + str(acrossWordNums[a]) + 'across'
        if x in downWordIDs:
            downNumber = ' ' + str(downWordNums[b]) + 'down'
        else:
            downNumber = b_alt

    # Save down number for later reference
    bstring.append(downNumber)

    # Select printed cell number
    if x in wordStartList:
        cellNumber = i
        i += 1
    else:
        cellNumber = ''
    
    # Create and label new cells
    divline = divline + ' id="' + str(x) + '" class="' + cellType + acrossNumber + downNumber + '">' + str(cellNumber)

    # Create and label new cell fills
    divline_fill = '<div id="fill_' + str(x) + '" class="userFill"></div>'

    # Reset
    grid_subdivs += '\n\t\t\t\t' + divline + '\n\t\t\t\t\t' + divline_fill + '\n\t\t\t\t</div>'
    divline = '<div'
    divline_fill = '<div'

In [ ]:
# Create across clues
divline_cluebox = '<div id="'
divline_indicator = '<div id="'
divline_cluenum = '<div id="'
divline_cluetext = '<div id="'
acrossClue_subdivs = ''
for x in range(1, len(acrossClues)+1):
    
    # Add clue boxes
    clueID = str(acrossWordNums[x-1]) + 'acrossClue'
    divline_cluebox = divline_cluebox + clueID + '" class="clueBox acrossClueBox">'

    # Add clue partial highlight
    divline_indicator = divline_indicator + clueID + '_INDICATOR" class="clueIndicator"></div>'

    # Add clue numbers
    divline_cluenum = divline_cluenum + clueID + '_NUM" class="clueNum">' + str(acrossWordNums[x-1]) + '</div>'

    # Add clue text
    divline_cluetext = divline_cluetext + clueID + '_TEXT" class="clueText">' + str(acrossClues[x-1]) + '</div>'

    # Reset
    acrossClue_subdivs += '\n\t\t\t\t\t\t' + divline_cluebox + '\n\t\t\t\t\t\t\t' + divline_indicator + '\n\t\t\t\t\t\t\t' + divline_cluenum + '\n\t\t\t\t\t\t\t' + divline_cluetext + '\n\t\t\t\t\t\t</div>'
    divline_cluebox = '<div id="'
    divline_indicator = '<div id="'
    divline_cluenum = '<div id="'
    divline_cluetext = '<div id="'

# Create down clues
divline_cluebox = '<div id="'
divline_indicator = '<div id="'
divline_cluenum = '<div id="'
divline_cluetext = '<div id="'
downClue_subdivs = ''
for x in range(1, len(downClues)+1):
    
    # Add clue boxes
    clueID = str(downWordNums[x-1]) + 'downClue'
    divline_cluebox = divline_cluebox + clueID + '" class="clueBox downClueBox">'

    # Add clue partial highlight
    divline_indicator = divline_indicator + clueID + '_INDICATOR" class="clueIndicator"></div>'

    # Add clue numbers
    divline_cluenum = divline_cluenum + clueID + '_NUM" class="clueNum">' + str(downWordNums[x-1]) + '</div>'

    # Add clue text
    divline_cluetext = divline_cluetext + clueID + '_TEXT" class="clueText">' + str(downClues[x-1]) + '</div>'

    # Reset
    downClue_subdivs += '\n\t\t\t\t\t\t' + divline_cluebox + '\n\t\t\t\t\t\t\t' + divline_indicator + '\n\t\t\t\t\t\t\t' + divline_cluenum + '\n\t\t\t\t\t\t\t' + divline_cluetext + '\n\t\t\t\t\t\t</div>'
    divline_cluebox = '<div id="'
    divline_indicator = '<div id="'
    divline_cluenum = '<div id="'
    divline_cluetext = '<div id="'

In [ ]:
html_content = '''<!DOCTYPE html>
<html lang="en-US">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <meta name="description" content="Play the daily crossword!">
        <title id="tabTitle">''' + full_date + '''</title>
        <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css">
        <link rel="stylesheet" href="https://fonts.googleapis.com/css?family=Fjalla+One|Libre+Baskerville">
        <link rel="stylesheet" href="gui.css">
    </head>
    <body>
		<style>
			:root {
				--n: ''' + str(n) + ''';
			}
			#grid {
				user-select: none;
				position: absolute;
				display: grid;
				grid-template-columns: repeat(var(--n), 1fr);
				grid-template-rows: repeat(var(--n), 1fr);
				width: 50%;
				border: 1px solid black;
				aspect-ratio: 1 / 1;
			}
		</style>
        <!--Header and Nav Bars-->
        <div id="pageHeader">
            <h1 id="pageTitle">Good Word</h1>
        </div>
        <div id="pageNavBar">
            <a id="puzzleHomeButton" class="button" href="https://jaedawoodruff.github.io/xwords/index.html">All Puzzles</a>
            <a id="aboutButton" class="button" href="https://jaedawoodruff.github.io/xwords/about.html">About</a>
        </div>
        <div id="puzzleInfo">
            <h2 id="puzzleTitle">''' + title + '''</h2>
            <h3 id="puzzleDate">''' + full_date + '''</h3>
            <h3 id="puzzleAuthor">''' + full_byline + '''</h3>
        </div>
        <div id="puzzleNavBar">
            <div id="pencilButton" class="fas fa-pencil button"></div>
            <div id="clearButton" class="button">Clear</div>
            <div id="checkButton" class="button">Check</div>
            <div id="checkMenu">
                <div id="checkLetterButton" class="checkSubButton button">Letter</div>
                <div id="checkWordButton" class="checkSubButton button">Word</div>
                <div id="checkGridButton" class="checkSubButton button">Grid</div>
                <div id="autocheckButton" class="checkSubButton button">Auto</div>
            </div>
            <div id="timer" class="button">00:00</div>
        </div>
        <!--Grid and Clues-->
        <div id="pageMain">
            <div id="grid">''' + grid_subdivs + '\n\t\t\t' + '''</div>
            <div id="cluesBucket">
                <div id="acrossCluesBucket" class="cluesBucketClass">
                    <div id="acrossCluesHeader" class="cluesHeader">Across</div>
                    <div id="acrossClues" class="clues">''' + acrossClue_subdivs + '\n\t\t\t\t\t' + '''</div>
                </div>
                <div id="downCluesBucket" class="cluesBucketClass">
                    <div id="downCluesHeader" class="cluesHeader">Down</div>
                    <div id="downClues" class="clues">''' + downClue_subdivs + '\n\t\t\t\t\t' + '''</div>
                </div>
            </div>
            <div id="ccPageMainBlur" class="pageMainBlur">
                <div id="ccPopupBox" class="popupBox">
                    <div id="ccPopupQuestion" class="popupQuestion"></div>
                    <div id="ccButtonsBox">
                        <div id="ccClear" class="ccButtons">Clear Puzzle</div>
                        <div id="ccCancel" class="ccButtons">Cancel</div>
                    </div>
                </div>
            </div>
            <div id="tmPageMainBlur" class="pageMainBlur">
                <div id="tmPopupBox" class="popupBox">
                    <div id="tmPopupQuestion" class="popupQuestion"></div>
                    <div id="tmContinue">Continue</div>
                </div>
            </div>
            <div id="stPageMainBlur" class="pageMainBlur">
                <div id="stPopupBox" class="popupBox">
                    <div id="stPopupQuestion" class="popupQuestion">Ready to play?</div>
                    <div id="stStart">Begin</div>
                </div>
            </div>
            <div id="footer"></div>
        </div>
        <!--Scripts-->
        <script src="xword_gui_functions.js"></script>
        <script>

            // PART 0: GET ELEMENTS

            // Puzzle specific variables
            var n = ''' + str(n) + ''';
            const blackCellIDs = ''' + str(blackCells) + ''';
            var acrossClues = ''' + str(acrossClues) + ''';
            var acrossWordNums = ''' + str(acrossWordNums) + ''';
            var downClues = ''' + str(downClues) + ''';
            var downWordNums = ''' + str(downWordNums) + ''';
            var downWordIDs = ''' + str(downWordIDs) + ''';
            var answers = "''' + answers + '''";
        
            // Global variables
            var newCell = null;
            var activeDirection = "across";
            var activeCellID = null;
            var clueID = null;
            var newClueBox = null;
            var initialActiveCellID = null;
            var autocheckOn = false;

            // Buttons and popups
            var newSTStart = document.getElementById("stStart");
            var newTimerButton = document.getElementById("timer");
            var timerPaused = true;
            var newTMPopupQuestion = document.getElementById("tmPopupQuestion");
            var newTMContinue = document.getElementById("tmContinue");
            var newPencilButton = document.getElementById("pencilButton");
            var newClearButton = document.getElementById("clearButton");
            var newCCPopupQuestion = document.getElementById("ccPopupQuestion");
            var newCCClear = document.getElementById("ccClear");
            var newCCCancel = document.getElementById("ccCancel");
            var newCheckButton = document.getElementById("checkButton");
            var newCheckMenu = document.getElementById("checkMenu");
            var newCheckLetterButton = document.getElementById("checkLetterButton");
            var newCheckWordButton = document.getElementById("checkWordButton");
            var newCheckGridButton = document.getElementById("checkGridButton");
            var newAutoCheckButton = document.getElementById("autocheckButton");

            /////////////////////////////////////

            // PART 1: HANDLE CELL CLICKS

            for (let x = 1; x < (n*n) + 1; x++) {
                if (!blackCellIDs.includes(x)) {

                    // Get elements
                    newCell = document.getElementById(x.toString());
                    
                    // Listen for clicks
                    newCell.addEventListener("click", function(event) {
                        
                        // If single click, highlight word
                        if (event.detail === 1) {
                            handleCellClick(this.id);
                        }

                        // If double click, switch direction and highlight new word
                        else if (event.detail === 2) {
                            switchActiveDirection();
                            handleCellClick(this.id);
                        }

                        // Save active cell ID
                        activeCellID = this.id;

                    });

                }
            }

            /////////////////////////////////////

            // PART 2: HANDLE CLUEBOX CLICKS

            // Add event listener for across clue boxes
            for (let x = 1; x < acrossClues.length + 1; x++) {

                // Get elements
                clueID = acrossWordNums[x-1].toString() + "acrossClue";
                newClueBox = document.getElementById(clueID);

                // Add event listener
                newClueBox.addEventListener("click", function() {
                    activeCellID = handleClueBoxClick(this.id);  
                });

            }

            // Add event listener for down clue boxes
            for (let x = 1; x < downClues.length + 1; x++) {

                // Get elements
                clueID = downWordNums[x-1].toString() + "downClue";
                newClueBox = document.getElementById(clueID);

                // Add event listener
                newClueBox.addEventListener("click", function() {
                    activeCellID = handleClueBoxClick(this.id);  
                });

            }

            /////////////////////////////////////

            // PART 3: HANDLE KEYSTROKES

            // Listen for keystrokes
            document.addEventListener("keydown", function(event) {

                // Move right
                if (event.key === "ArrowRight") {
                    event.preventDefault();
                    if (activeDirection === "across") {
                        activeCellID = moveRight(activeCellID);
                        handleCellClick(activeCellID);
                    }
                    else {
                        switchActiveDirection();
                        activeCellID = moveLeft(activeCellID);
                    }
                }

                // Move left
                else if (event.key === "ArrowLeft") {
                    event.preventDefault();
                    if (activeDirection === "across") {
                        activeCellID = moveLeft(activeCellID);
                        handleCellClick(activeCellID);
                    }
                    else {
                        switchActiveDirection();
                        activeCellID = moveRight(activeCellID);
                    }
                }

                // Move up
                else if (event.key === "ArrowUp") {
                    event.preventDefault();
                    if (activeDirection === "down") {
                        activeCellID = moveUp(activeCellID);
                        handleCellClick(activeCellID);
                    }
                    else {
                        switchActiveDirection();
                        activeCellID = moveDown(activeCellID);
                    }
                }

                // Move down
                else if (event.key === "ArrowDown") {
                    event.preventDefault();
                    if (activeDirection === "down") {
                        activeCellID = moveDown(activeCellID);
                        handleCellClick(activeCellID);
                    }
                    else {
                        switchActiveDirection();
                        activeCellID = moveUp(activeCellID);
                    }
                }

                
                // If click backspace, clear cell and move back a space
                else if (event.key === "Backspace") {

                    // Fill cell with blank
                    document.getElementById("fill_" + activeCellID).textContent = "";
                    
                    // Move back
                    if (activeDirection === "across") {
                        activeCellID = moveLeft(activeCellID);
                    }
                    else {
                        activeCellID = previousDownLetter(activeCellID, downWordNums, downWordIDs);
                    }

                    // Handle cell click
                    handleCellClick(activeCellID);
                }

                // If it's enter or tab, move to next word
                else if (event.key === "Enter" || event.key === "Tab") {
                    event.preventDefault();
                    activeCellID = nextWord(activeCellID, downWordNums, downWordIDs);
                    handleCellClick(activeCellID);
                }

                // If it's a spacebar, do not autoscroll
                else if (event.key === " ") {
                    event.preventDefault();
                    activeCellID = fillCell(activeCellID, event.key, downWordNums, downWordIDs);
                    handleCellClick(activeCellID);
                }

                // If it's a letter/digit key, fill cell
                else if (event.key.length === 1) {

                    // Fill cell
                    initialActiveCellID = activeCellID;
                    activeCellID = fillCell(activeCellID, event.key, downWordNums, downWordIDs);
                    handleCellClick(activeCellID);

                    // Autocheck if on!
                    if (autocheckOn === true) {
                        checkGrid(answers);
                    }

                    // Update gray cells
                    grayClues(initialActiveCellID, activeDirection);

                }

            });

            /////////////////////////////////////

            // PART 4: OTHER EVENT LISTENERS

            // Start puzzle popup
            newSTStart.addEventListener("click", function () {
                hideDiv("stPageMainBlur");
                hideDiv("stStart");
                timerPaused = false;
            });

            // Start timer
            incrementTimer();

            // Timer button
            newTimerButton.addEventListener("click", function () {
                if (timerPaused === false) {
                    // Show new popup
                    showDiv("tmPageMainBlur");
                    showDiv("tmContinue");
                    newTMPopupQuestion.textContent = "Timer paused. Want to continue?";
                    // Hide old popups
                    hideDiv("ccPageMainBlur");
                    hideDiv("ccButtonsBox");
                    timerPaused = true;
                }
            });
            newTMContinue.addEventListener("click", function () {
                hideDiv("tmPageMainBlur");
                hideDiv("tmContinue");
                timerPaused = false;
            });

            // Pencil button
            newPencilButton.addEventListener("click", function () {
                togglePencil();
            });

            // Clear button
            newClearButton.addEventListener("click", function () {
                if (timerPaused === false) {
                    // Show new popup
                    showDiv("ccPageMainBlur");
                    showDiv("ccButtonsBox");
                    newCCPopupQuestion.textContent = "Are you sure you want to clear the puzzle?";
                    // Hide old popups
                    hideDiv("tmPageMainBlur");
                    hideDiv("tmContinue");
                }
            });
            // Confirm clear
            newCCClear.addEventListener("click", function () {
                clearGrid();
                hideDiv("ccPageMainBlur");
                hideDiv("ccButtonsBox");
            });
            // Confirm clear cancel
            newCCCancel.addEventListener("click", function () {
                hideDiv("ccPageMainBlur");
                hideDiv("ccButtonsBox");
            });

            // Check button
            newCheckButton.addEventListener("mouseenter", function () {
                showDiv("checkMenu");
            });
            newCheckButton.addEventListener("mouseleave", function () {
                hideDiv("checkMenu");
            });
            // Check menu
            newCheckMenu.addEventListener("mouseenter", function () {
                showDiv("checkMenu");
            });
            newCheckMenu.addEventListener("mouseleave", function () {
                hideDiv("checkMenu");
            });
            
            // Check letter
            newCheckLetterButton.addEventListener("click", function () {
                checkCell(activeCellID, answers);
            });
            // Check word
            newCheckWordButton.addEventListener("click", function () {
                checkWord(activeCellID, answers);
            });
            // Check grid
            newCheckGridButton.addEventListener("click", function () {
                checkGrid(answers);
            });

            // Autocheck command (click)
            newAutoCheckButton.addEventListener("click", function () {
                if (autocheckOn === false) {
                    autocheckOn = true;
                    newAutoCheckButton.style.backgroundColor = "lightblue";
                    checkGrid(answers);
                }
                else {
                    autocheckOn = false;
                    newAutoCheckButton.style.backgroundColor = "transparent";
                }
            });
            // Autocheck command (hover)
            newAutoCheckButton.addEventListener("mouseenter", function () {
                newAutoCheckButton.style.backgroundColor = "lightgray";
            });
            newAutoCheckButton.addEventListener("mouseleave", function () {
                if (autocheckOn === true) {
                    newAutoCheckButton.style.backgroundColor = "lightblue";
                }
                else {
                    newAutoCheckButton.style.backgroundColor = "transparent";
                }
            });

        </script>
        <div id="fullPageBlur">Please adjust window size or zoom level to play.</div>
    </body>
</html>'''

In [ ]:
# Print sanity checks

print("PUZZLE INFO")
print("-----------")
print(str(n) + " x " + str(n))
print(title)
print(full_date)
print(full_byline)
print("")
print("")

print("FIRST AND LAST ACROSS CLUES")
print("---------------------------")
print(answers.split(".")[0], ":", acrossClues[0])
print(answers.split(".")[-1], ":", acrossClues[-1])
print("")
print("")

print("NUMBER LISTS")
print("------------")
print("acrossWordNums:", acrossWordNums)
print("downWordNums:", downWordNums)
print("blackCells:", blackCells)
print("downWordIDs:", downWordIDs)


In [ ]:
# Save new month-DD-YYYY.html file
with open(filename, "w", encoding="utf-8") as file:
    file.write(html_content)

In [ ]:
# Prep .html auto-updates

# Import libraries
from bs4 import BeautifulSoup
import re
from datetime import datetime

# Get new today link
newTodayLink = "https://jaedawoodruff.github.io/xwords/" + filename

# Get new puzzle info
newDate = weekday + " " + date
newTitle = '"' + title + '"'
newAuthor = "By " + author

In [ ]:
# Update about.html

# Read about.html file
filepath_about = r"C:\Users\Jaeda\OneDrive\Documents\xword code\about.html"
with open(filepath_about, "r", encoding="utf-8") as f:
    data_about = f.read()

# Find old today link + replace with new today link
soup = BeautifulSoup(data_about, "html.parser")
#oldTodayLink = soup.find("a", id="homeButton").get("href")
data_about = re.sub(oldTodayLink, newTodayLink, data_about)

# Save new about.html file
with open(filepath_about, "w", encoding="utf-8") as file:
    file.write(data_about)

In [ ]:
# Update index.html

# Read index.html file
filepath_index = r"C:\Users\Jaeda\OneDrive\Documents\xword code\index.html"
with open(filepath_index, "r", encoding="utf-8") as f:
    data_index = f.read()

# Read old puzzle info (archived puzzles)
soup = BeautifulSoup(data_index, "html.parser")
oldDate = soup.find("div", id="todayPuzzleDate").__getattribute__("text").strip()
oldTitle = soup.find("div", id="todayPuzzleTitle").__getattribute__("text").strip()
oldAuthor = soup.find("div", id="todayPuzzleAuthor").__getattribute__("text").strip()
oldTodayLinkMainButton = soup.find("a", id="todayPuzzleMainButton").get("href")

# Create new archived puzzle cell
olderPuzzleBucket = soup.find("div", id="olderPuzzleBucket")
newArchivePuzzle = soup.new_tag("a", href=oldTodayLinkMainButton, **{"class": "olderPuzzle"})
olderPuzzleBucket.insert(0, newArchivePuzzle)
# New archive puzzle date
newArchivePuzzleDate = soup.new_tag("div", **{"class": "puzzleDate"})
newArchivePuzzle.append(newArchivePuzzleDate)
monthNum = datetime.strptime(oldDate.split(" ")[1], '%B').month
oldDateFormatted = oldDate.split(" ")[0] + " " + str(monthNum) + "/" + oldDate.split(" ")[2][:-1]
newArchivePuzzleDate.string = oldDateFormatted
# New archive puzzle title
newArchivePuzzleTitle = soup.new_tag("div", **{"class": "puzzleTitle"})
newArchivePuzzle.append(newArchivePuzzleTitle)
newArchivePuzzleTitle.string = oldTitle
# New archive puzzle author
newArchivePuzzleAuthor = soup.new_tag("div", new_content=oldAuthor, **{"class": "puzzleAuthor"})
newArchivePuzzle.append(newArchivePuzzleAuthor)
newArchivePuzzleAuthor.string = oldAuthor

# Read old puzzle info (main puzzle block)
oldDate = soup.find("div", id="todayPuzzleDate")
oldTitle = soup.find("div", id="todayPuzzleTitle")
oldAuthor = soup.find("div", id="todayPuzzleAuthor")
oldTodayLinkMainButton = soup.find("a", id="todayPuzzleMainButton")

# Update today puzzle cell (date, title, author, today puzzle link, today navbar link)
oldDate.string = newDate
oldTitle.string = newTitle
oldAuthor.string = newAuthor
oldTodayLinkMainButton["href"] = newTodayLink

# Save new index.html file
with open(filepath_index, "w", encoding="utf-8") as file:
    file.write(soup.prettify())

In [ ]:
# Upload all the open tabs here to github including the ipynb

# Try quick autopublish via github? I.e. have .py script on the github page and have it publish new html files directly to github? If possible?
# Write out workflow: run .py file, upload new files to github (march-23-2026.html, gui.css, about.html) (index.html???)
# Clean up file (e.g. move all import lines to the top, combine cells, etc)
# Convert .ipynb to .py file

# Upload one puzzle to github using .py file and workflow + a few diff cfp files
# Upload another
# Upload another

# How to make my code immune to some rando beautifulsouping it?
# Block web scraping?

# Note: index.html updates depend on the index.html file being in the same location! (hm or move this up to optional settings at the top? yeah i think thats good)
